In [ ]:
## INFORMATIONS PACKAGE - CITIES HTML DOWNLOADER
## Download Wikipedia HTML files for all Romanian cities into 'informations' package

In [1]:
## DOWNLOAD ROMANIAN CITIES HTML FILES INTO 'informations' PACKAGE

import os
import json
import requests
import time
from pathlib import Path
from urllib.parse import quote
import hashlib

# Create 'informations' package directory structure
informations_dir = Path('informations')
informations_dir.mkdir(exist_ok=True)

# Create __init__.py to make it a package
init_file = informations_dir / '__init__.py'
init_file.write_text('"""Romanian Cities Information Package"""\n')

# Create cities subdirectory
cities_dir = informations_dir / 'cities'
cities_dir.mkdir(exist_ok=True)

print("📦 Setting up 'informations' package structure...")
print(f"   ✓ Package directory: {informations_dir.absolute()}")
print(f"   ✓ Cities HTML directory: {cities_dir.absolute()}\n")

# Load city data from JSON
json_file = 'romanian_cities_complete.json'
if not os.path.exists(json_file):
    print(f"⚠ Warning: {json_file} not found!")
    print("  Make sure you've run the web scraping cell in L2.ipynb first.")
    cities_data = []
else:
    with open(json_file, 'r', encoding='utf-8') as f:
        cities_data = json.load(f)
    print(f"✓ Loaded {len(cities_data)} cities from {json_file}\n")

# Setup session with proper headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Romanian-Cities-Bot/1.0 (Educational; +https://github.com/your-repo)',
    'Accept-Encoding': 'gzip'
})

# Download HTML for each city
print("=" * 80)
print("DOWNLOADING HTML FILES FOR ALL CITIES")
print("=" * 80)

downloaded_count = 0
failed_cities = []
total = len(cities_data)

for idx, city in enumerate(cities_data, 1):
    city_name = city.get('name', 'Unknown')
    
    # Sanitize filename (remove special characters)
    safe_filename = city_name.replace('/', '_').replace('\\', '_').replace(' ', '_').lower()
    html_file = cities_dir / f"{safe_filename}.html"
    
    # Show progress
    if idx % 10 == 0 or idx == 1:
        print(f"[{idx}/{total}] 📍 {city_name}...", end=" ", flush=True)
    else:
        print(".", end="", flush=True)
    
    try:
        # Construct REST API URL
        encoded_name = quote(city_name)
        rest_api_url = f"https://ro.wikipedia.org/w/rest.php/v1/page/{encoded_name}/html?redirect=no"
        
        # Download HTML
        response = session.get(rest_api_url, timeout=10)
        
        # Handle rate limiting
        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After', 5)
            wait_time = int(retry_after) if str(retry_after).isdigit() else 5
            print(f"\n⏳ Rate limited. Waiting {wait_time}s...")
            time.sleep(wait_time)
            response = session.get(rest_api_url, timeout=10)
        
        if response.status_code == 404:
            failed_cities.append((city_name, "Not found"))
            if idx % 10 == 0 or idx == 1:
                print("⊘ (404)")
        elif response.status_code == 200:
            # Save HTML file
            with open(html_file, 'w', encoding='utf-8') as f:
                f.write(response.text)
            downloaded_count += 1
            if idx % 10 == 0 or idx == 1:
                print(f"✓ ({len(response.text)} bytes)")
        else:
            failed_cities.append((city_name, f"HTTP {response.status_code}"))
            if idx % 10 == 0 or idx == 1:
                print(f"⚠ ({response.status_code})")
    
    except Exception as e:
        failed_cities.append((city_name, str(e)))
        if idx % 10 == 0 or idx == 1:
            print(f"✗ ({e})")
    
    # Respect rate limits: ~5 requests/sec
    time.sleep(0.2)

print("\n" + "=" * 80)
print(f"✓ Successfully downloaded {downloaded_count}/{total} HTML files")
if failed_cities:
    print(f"⚠ Failed: {len(failed_cities)} cities")
    for city_name, reason in failed_cities[:5]:
        print(f"   - {city_name}: {reason}")
print("=" * 80)

# Create metadata file
metadata = {
    'total_cities': total,
    'downloaded': downloaded_count,
    'failed': len(failed_cities),
    'cities_downloaded': [(c.get('name'), safe_filename := c.get('name').replace('/', '_').replace('\\', '_').replace(' ', '_').lower()) for c in cities_data if c.get('name')],
    'failed_cities': failed_cities
}

metadata_file = informations_dir / 'metadata.json'
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"\n✓ Metadata saved to: {metadata_file}")

# Create index file
index_file = informations_dir / 'INDEX.md'
index_content = f"""# Romanian Cities Information Package

## Overview
- Total cities: {total}
- Successfully downloaded: {downloaded_count}
- Failed: {len(failed_cities)}

## Directory Structure
```
informations/
├── __init__.py           # Package initialization
├── cities/               # HTML files for each city
│   ├── bucuresti.html
│   ├── cluj-napoca.html
│   └── ...
├── metadata.json         # Download statistics
└── INDEX.md             # This file
```

## Usage
```python
from pathlib import Path

# Load a city's HTML
cities_dir = Path('informations/cities')
bucharest_html = (cities_dir / 'bucuresti.html').read_text(encoding='utf-8')

# Parse with BeautifulSoup
from bs4 import BeautifulSoup
soup = BeautifulSoup(bucharest_html, 'html.parser')
```

## Files Downloaded
{chr(10).join(f"- {c.get('name')}: `cities/{c.get('name').replace('/', '_').replace(chr(92), '_').replace(' ', '_').lower()}.html`" for c in cities_data[:10])}
... and {len(cities_data) - 10} more cities
"""

with open(index_file, 'w', encoding='utf-8') as f:
    f.write(index_content)

print(f"✓ Index created: {index_file}")

# Summary
print("\n" + "=" * 80)
print("📦 PACKAGE CREATED SUCCESSFULLY!")
print("=" * 80)
print(f"\nLocation: {informations_dir.absolute()}")
print(f"\nStructure:")
print(f"  informations/")
print(f"  ├── __init__.py")
print(f"  ├── metadata.json")
print(f"  ├── INDEX.md")
print(f"  └── cities/ ({downloaded_count} HTML files)")
print(f"\nImport in other scripts:")
print(f"  from informations.cities import *")
print(f"  # Or load HTML files programmatically")

📦 Setting up 'informations' package structure...
   ✓ Package directory: c:\Users\Alex\Documents\llms\project\informations
   ✓ Cities HTML directory: c:\Users\Alex\Documents\llms\project\informations\cities

✓ Loaded 315 cities from romanian_cities_complete.json

DOWNLOADING HTML FILES FOR ALL CITIES
[1/315] 📍 București... ✓ (1130986 bytes)
........[10/315] 📍 Ploiești... ✓ (343421 bytes)
.........[20/315] 📍 Satu Mare... ✓ (399026 bytes)
.........[30/315] 📍 Alba Iulia... ✓ (347593 bytes)
.........[40/315] 📍 Hunedoara... ✓ (248963 bytes)
.........[50/315] 📍 Medgidia... ✓ (134999 bytes)
.........[60/315] 📍 Petroșani... ✓ (181784 bytes)
.........[70/315] 📍 Caracal... ✓ (245918 bytes)
.........[80/315] 📍 Roșiorii de Vede... ✓ (1411 bytes)
.........[90/315] 📍 Comănești... ✓ (149092 bytes)
.........[100/315] 📍 Blaj... ✓ (210018 bytes)
.........[110/315] 📍 Salonta... ✓ (160499 bytes)
.........[120/315] 📍 Chitila... ✓ (123413 bytes)
.........[130/315] 📍 Urziceni... ✓ (156945 bytes)
.........[1

In [2]:
## LOAD AND PARSE HTML FILES FROM INFORMATIONS PACKAGE

from pathlib import Path
from bs4 import BeautifulSoup
import json

# Load metadata
informations_dir = Path('informations')
metadata_file = informations_dir / 'metadata.json'

with open(metadata_file, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"📦 Package Statistics:")
print(f"   Total cities indexed: {metadata['total_cities']}")
print(f"   HTML files downloaded: {metadata['downloaded']}")
print(f"   Failed downloads: {metadata['failed']}\n")

# Example: Load Bucharest HTML
cities_dir = informations_dir / 'cities'
bucharest_file = cities_dir / 'bucuresti.html'

if bucharest_file.exists():
    print("📖 Example: Loading Bucharest HTML\n")
    
    # Read HTML
    html_content = bucharest_file.read_text(encoding='utf-8')
    print(f"   HTML file size: {len(html_content)} bytes")
    
    # Parse with BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Extract info
    print(f"   Parsed successfully with BeautifulSoup")
    
    # Find infobox
    infobox = soup.find('table', {'class': ['infobox', 'infobox-settlement']})
    if infobox:
        print(f"   ✓ Found infobox table")
        print(f"   Infobox rows: {len(infobox.find_all('tr'))}")
    
    # Find paragraphs
    paragraphs = soup.find_all('p')
    print(f"   ✓ Found {len(paragraphs)} paragraphs")
    
    if len(paragraphs) > 0:
        first_para = paragraphs[0].get_text()[:100]
        print(f"   📄 First paragraph: {first_para}...\n")

# List all downloaded cities
print("=" * 80)
print("📍 AVAILABLE CITIES (first 20):")
print("=" * 80)

html_files = sorted(cities_dir.glob('*.html'))
for i, html_file in enumerate(html_files[:20], 1):
    file_size = html_file.stat().st_size
    print(f"{i:2d}. {html_file.stem:<25} ({file_size:>7,} bytes)")

if len(html_files) > 20:
    print(f"... and {len(html_files) - 20} more cities")

print(f"\nTotal HTML files available: {len(html_files)}")

📦 Package Statistics:
   Total cities indexed: 315
   HTML files downloaded: 315
   Failed downloads: 0

📍 AVAILABLE CITIES (first 20):
 1. abrud                     (170,665 bytes)
 2. adjud                     (179,199 bytes)
 3. agnita                    (163,733 bytes)
 4. aiud                      (198,626 bytes)
 5. alba_iulia                (353,378 bytes)
 6. alexandria                (  1,407 bytes)
 7. aleșd                     (168,408 bytes)
 8. amara                     (221,589 bytes)
 9. anina                     (141,528 bytes)
10. aninoasa                  (123,406 bytes)
11. arad                      (439,922 bytes)
12. ardud                     (117,424 bytes)
13. avrig                     (444,804 bytes)
14. azuga                     (166,513 bytes)
15. babadag                   (120,431 bytes)
16. bacău                     (417,828 bytes)
17. baia_de_aramă             (144,774 bytes)
18. baia_de_arieș             (139,402 bytes)
19. baia_mare                 (442,0

In [3]:
## BATCH EXTRACT DATA FROM ALL CITY HTML FILES

from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd

def extract_city_info_from_html(html_file):
    """Extract structured data from a city's HTML file"""
    try:
        html_content = html_file.read_text(encoding='utf-8')
        soup = BeautifulSoup(html_content, 'html.parser')
        
        city_name = html_file.stem.replace('_', ' ').title()
        
        info = {
            'city': city_name,
            'html_file': html_file.name,
            'file_size': html_file.stat().st_size,
            'has_infobox': bool(soup.find('table', {'class': ['infobox', 'infobox-settlement']})),
            'paragraphs': len(soup.find_all('p')),
            'headings': len(soup.find_all(['h2', 'h3'])),
            'tables': len(soup.find_all('table')),
            'links': len(soup.find_all('a')),
        }
        
        # Try to extract first paragraph
        paragraphs = soup.find_all('p')
        if paragraphs:
            info['first_paragraph'] = paragraphs[0].get_text()[:150]
        
        return info
    except Exception as e:
        return {'city': html_file.stem, 'error': str(e)}

# Process all HTML files
print("=" * 100)
print("EXTRACTING DATA FROM ALL CITY HTML FILES")
print("=" * 100)

cities_dir = Path('informations/cities')
html_files = sorted(cities_dir.glob('*.html'))

all_info = []
for idx, html_file in enumerate(html_files, 1):
    if idx % 20 == 0 or idx == 1:
        print(f"[{idx}/{len(html_files)}] Processing {html_file.stem}...", end=" ", flush=True)
    else:
        print(".", end="", flush=True)
    
    info = extract_city_info_from_html(html_file)
    all_info.append(info)
    
    if idx % 20 == 0 or idx == 1:
        print(f"✓")

print("\n" + "=" * 100)
print(f"✓ Extracted data from {len(all_info)} cities\n")

# Create summary dataframe
df_summary = pd.DataFrame(all_info)

# Display statistics
print("📊 CONTENT STATISTICS:")
print("-" * 100)
print(f"Average HTML file size: {df_summary['file_size'].mean():,.0f} bytes")
print(f"Average paragraphs per city: {df_summary['paragraphs'].mean():.1f}")
print(f"Average headings per city: {df_summary['headings'].mean():.1f}")
print(f"Average tables per city: {df_summary['tables'].mean():.1f}")
print(f"Average links per city: {df_summary['links'].mean():.1f}")
print(f"Cities with infobox: {df_summary['has_infobox'].sum()}/{len(df_summary)}")

# Show top cities by HTML size
print("\n🔝 TOP 10 CITIES BY HTML SIZE:")
print("-" * 100)
top_cities = df_summary.nlargest(10, 'file_size')[['city', 'file_size', 'paragraphs', 'tables']]
for idx, (i, row) in enumerate(top_cities.iterrows(), 1):
    print(f"{idx:2d}. {row['city']:<25} {row['file_size']:>10,} bytes | {row['paragraphs']:>3} paragraphs | {row['tables']:>2} tables")

# Save summary to CSV
summarexy_file = Path('informations/cities_summary.csv')
df_summary.to_csv(summary_file, index=False, encoding='utf-8')
print(f"\n✓ Summary saved to: {summary_file}")

# Display sample first paragraphs
print("\n📖 SAMPLE CITY DESCRIPTIONS (first paragraph):")
print("=" * 100)
for idx, row in df_summary[df_summary['first_paragraph'].notna()].head(5).iterrows():
    print(f"\n{row['city']}:")
    print(f"  {row['first_paragraph']}...")

EXTRACTING DATA FROM ALL CITY HTML FILES
[1/315] Processing abrud... ✓
..................[20/315] Processing baia_sprie... ✓
...................[40/315] Processing breaza... ✓
...................[60/315] Processing băilești... ✓
...................[80/315] Processing copșa_mică... ✓
...................[100/315] Processing deta... ✓
...................[120/315] Processing făget... ✓
...................[140/315] Processing ianca... ✓
...................[160/315] Processing miercurea_sibiului... ✓
...................[180/315] Processing nădlac... ✓
...................[200/315] Processing petrila... ✓
...................[220/315] Processing roznov... ✓
...................[240/315] Processing sinaia... ✓
...................[260/315] Processing săveni... ✓
...................[280/315] Processing târgu_ocna... ✓
...................[300/315] Processing voluntari... ✓
...............
✓ Extracted data from 315 cities

📊 CONTENT STATISTICS:
--------------------------------------------------------

In [4]:
## EXTRACT PLAIN TEXT FROM HTML FILES

from pathlib import Path
from bs4 import BeautifulSoup
import re

def extract_text_from_html(html_file):
    """Extract clean text content from HTML file"""
    try:
        html_content = html_file.read_text(encoding='utf-8')
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(['script', 'style']):
            script.decompose()
        
        # Get text
        text = soup.get_text()
        
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = '\n'.join(chunk for chunk in chunks if chunk)
        
        return text
    except Exception as e:
        return f"Error extracting text: {str(e)}"

# Create output directory
print("=" * 100)
print("EXTRACTING PLAIN TEXT FROM ALL HTML FILES")
print("=" * 100)

cities_dir = Path('informations/cities')
text_dir = Path('informations/cities_text')
text_dir.mkdir(exist_ok=True)

print(f"\n📁 Output directory: {text_dir.absolute()}\n")

html_files = sorted(cities_dir.glob('*.html'))
success_count = 0
error_count = 0

for idx, html_file in enumerate(html_files, 1):
    # Show progress
    if idx % 20 == 0 or idx == 1:
        print(f"[{idx}/{len(html_files)}] Processing {html_file.stem}...", end=" ", flush=True)
    else:
        print(".", end="", flush=True)
    
    # Extract text
    text_content = extract_text_from_html(html_file)
    
    # Save to text file
    text_file = text_dir / f"{html_file.stem}.txt"
    try:
        text_file.write_text(text_content, encoding='utf-8')
        success_count += 1
        if idx % 20 == 0 or idx == 1:
            print(f"✓ ({len(text_content)} chars)")
    except Exception as e:
        error_count += 1
        if idx % 20 == 0 or idx == 1:
            print(f"✗ (Error: {e})")

print("\n" + "=" * 100)
print(f"✓ EXTRACTION COMPLETE!")
print("-" * 100)
print(f"Successfully created: {success_count} text files")
if error_count > 0:
    print(f"Errors: {error_count}")
print("=" * 100)

# Show statistics
print("\n📊 TEXT FILES STATISTICS:")
print("-" * 100)

text_files = sorted(text_dir.glob('*.txt'))
file_sizes = [f.stat().st_size for f in text_files]

print(f"Total text files: {len(text_files)}")
print(f"Average file size: {sum(file_sizes) / len(file_sizes):,.0f} bytes")
print(f"Largest file: {max(file_sizes):,} bytes")
print(f"Smallest file: {min(file_sizes):,} bytes")
print(f"Total storage: {sum(file_sizes):,} bytes ({sum(file_sizes) / (1024*1024):.1f} MB)")

# Show sample
print("\n📄 SAMPLE TEXT FILES (first 3):")
print("=" * 100)
for i, txt_file in enumerate(text_files[:3], 1):
    content = txt_file.read_text(encoding='utf-8')
    city_name = txt_file.stem
    preview = content[:200].replace('\n', ' ')
    print(f"\n{i}. {city_name}.txt ({len(content)} chars)")
    print(f"   Preview: {preview}...")

print(f"\n✓ All {len(text_files)} plain text files saved to: {text_dir}")

EXTRACTING PLAIN TEXT FROM ALL HTML FILES

📁 Output directory: c:\Users\Alex\Documents\llms\project\informations\cities_text

[1/315] Processing abrud... ✓ (15767 chars)
..................[20/315] Processing baia_sprie... ✓ (17966 chars)
...................[40/315] Processing breaza... ✓ (37668 chars)
...................[60/315] Processing băilești... ✓ (14649 chars)
...................[80/315] Processing copșa_mică... ✓ (26887 chars)
...................[100/315] Processing deta... ✓ (16226 chars)
...................[120/315] Processing făget... ✓ (13925 chars)
...................[140/315] Processing ianca... ✓ (10926 chars)
...................[160/315] Processing miercurea_sibiului... ✓ (21331 chars)
...................[180/315] Processing nădlac... ✓ (14954 chars)
...................[200/315] Processing petrila... ✓ (12123 chars)
...................[220/315] Processing roznov... ✓ (10144 chars)
...................[240/315] Processing sinaia... ✓ (16923 chars)
...................[260/